# 18 · Polarization multiplexing and MIMO equalization

A single-mode fiber guides two orthogonal polarizations, and coherent
systems send an independent signal on each, doubling the capacity.  The
fiber does not keep them apart: it rotates the state of polarization (SOP)
randomly and slowly, and polarization-mode dispersion (PMD) delays the two
principal states against each other.  The receiver sees a **2x2 MIMO**
mixture and must unmix it.

1. polarization rotation and the Jones-matrix model;
2. the **butterfly** equalizer: four adaptive filters;
3. **PMD**: a frequency-dependent rotation, and how many taps it needs;
4. tracking a **drifting** SOP;
5. blind unmixing and the channel **permutation**.

The data are `(2, N)` arrays: X and Y polarization on the first axis.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import commkit as ck
from commkit import RRC, Constellation
from commkit.backend import to_device
from commkit.recovery import DataAided

DEVICE = "gpu" if ck.backend.is_cupy_available() else "cpu"
print(DEVICE)

ck.plotting.apply_default_theme()
C16 = Constellation.qam(16)
TRAIN = 4000


def host(a):
    return to_device(a, "cpu")


tx = ck.generate(
    C16,
    num_symbols=2**15,
    symbol_rate=32e9,
    sps=2,
    pulse=RRC(rolloff=0.1),
    num_channels=2,
    rng=1,
    device=DEVICE,
)


def receive(x, rng=2):
    """AWGN at Es/N0 = 20 dB, then the matched filter."""
    return ck.filtering.matched_filter(
        ck.impairments.apply_awgn(x, esn0_db=20, rng=rng)
    )


def train(x, *, num_taps=15, step_size=3e-3):
    return ck.equalization.lms(
        x,
        num_taps=num_taps,
        step_size=step_size,
        training_symbols=x.reference.symbols[..., :TRAIN],
    )


def snr(result):
    return np.round(ck.metrics.snr(result.signal, num_skip_symbols=2 * TRAIN), 2)

## 1. Polarization rotation

A lossless fiber section acts on the field vector `[E_x, E_y]` as a
unitary 2x2 **Jones matrix**.  Its simplest form is a rotation by `θ`:

$$\begin{bmatrix} E_x' \\ E_y' \end{bmatrix} =
\begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix}
\begin{bmatrix} E_x \\ E_y \end{bmatrix}.$$

Each received polarization is then a mixture of both transmitted signals:
at 35° the constellation of each output is a blur of two overlapping 16-QAMs.

In [ ]:
mixed = receive(ck.impairments.apply_polarization_mixing(tx, theta=np.deg2rad(35)))
y = ck.multirate.decimate_to_symbol_rate(mixed)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for p, ax in enumerate(axes):
    ck.plotting.plot_constellation(
        y.samples[p], ax=ax, title=f"Output {'XY'[p]}, 35° rotation"
    )
plt.show()
print("SNR per output before unmixing:", np.round(ck.metrics.snr(y), 1), "dB")

## 2. The butterfly equalizer

The equalizers of [16](16-adaptive-equalization.ipynb) take `(2, N)` input and
become **butterflies**: each output is the sum of two FIR filters, one per
input,

$$y_x = \mathbf{w}_{xx}^H \mathbf{x}_x + \mathbf{w}_{xy}^H \mathbf{x}_y, \qquad
y_y = \mathbf{w}_{yx}^H \mathbf{x}_x + \mathbf{w}_{yy}^H \mathbf{x}_y.$$

A static rotation needs only one tap per filter (a matrix inverse); the
taps in time cover what the channel spreads over time.  The weights come
back as `(2, 2, num_taps)`.

In [ ]:
res = train(mixed)
print("SNR per polarization after the 2x2 LMS:", snr(res), "dB")
w = host(res.weights)
energy = np.sum(np.abs(w) ** 2, axis=-1)  # per filter, summed over the taps
print(
    "weights:",
    w.shape,
    "| tap energy per filter:\n",
    np.round(energy / energy.max(), 3),
)

All four filters are active: the cross filters take from each input the
part that belongs to the other polarization.

## 3. Polarization-mode dispersion

Fiber birefringence gives the two **principal states of polarization**
(PSPs) slightly different group velocities.  The **differential group
delay** (DGD) `τ` delays one against the other:

$$H(f) = R(\theta)\, \mathrm{diag}\!\left(e^{-j\pi f \tau}, e^{+j\pi f\tau}\right) R(-\theta),$$

a rotation that changes with frequency.  A one-tap matrix cannot undo it;
the filters need a time span of at least `τ`.  At 32 GBd one symbol is
31 ps, and 2 samples per symbol make `num_taps / 2` symbols of span.

In [ ]:
taps_grid = (3, 7, 15, 31)
print("DGD [ps]  " + "  ".join(f"{t:2d} taps" for t in taps_grid))
for dgd in (0.0, 10e-12, 30e-12, 60e-12):
    x = ck.impairments.apply_polarization_mixing(tx, theta=np.deg2rad(35))
    if dgd:
        x = ck.impairments.apply_pmd(x, dgd=dgd, theta=np.deg2rad(20))
    x = receive(x)
    row = [f"{float(np.mean(snr(train(x, num_taps=t)))):7.2f}" for t in taps_grid]
    print(f"{dgd * 1e12:6.0f}    " + "  ".join(row))

A 60 ps DGD, two symbol periods, defeats 3 taps (1.5 symbols) and is
undone completely with 7.  The mean DGD of a link grows with the square root
of its length (`pmd_ps_sqrt_km · sqrt(L)`), and its instantaneous value is
Maxwellian-distributed, so equalizers are sized with margin.  The
distributed form is part of the fiber model of
[20](20-nonlinear-fiber-and-backpropagation.ipynb) (`Fiber(pmd_ps_sqrt_km=...)`).

## 4. A drifting SOP

The SOP changes as the fiber moves and its temperature changes; in aerial
cables it can rotate at kilo- to mega-radians per second.  The equalizer
tracks it only if its step size allows: a small `μ` averages noise but
lags behind the rotation.  `drift_rate` (rad/s) rotates the SOP linearly
over the record.

In [ ]:
print("drift [Mrad/s]  " + "  ".join(f"μ = {mu:g}" for mu in (3e-4, 3e-3, 1e-2)))
for rate in (64e3, 640e3, 3.2e6):
    x = receive(
        ck.impairments.apply_polarization_mixing(tx, theta=0.3, drift_rate=rate)
    )
    print(
        f"{rate / 1e6:14.3f}  "
        + "  ".join(
            f"{float(np.mean(snr(train(x, step_size=mu)))):9.2f}"
            for mu in (3e-4, 3e-3, 1e-2)
        )
    )

## 5. Blind unmixing and the permutation

A blind equalizer (CMA, RDE, [17](17-blind-equalization.ipynb)) separates the two
polarizations without knowing which transmitted stream is which.  It can
converge to either solution: X on output X, or **swapped**.  It also leaves
each output with an arbitrary constant phase.  Metrics never guess
([06](06-awgn-and-performance-metrics.ipynb) · section 6), so the receiver resolves both against
known data: `resolve_channel_permutation` finds the output order,
`DataAided()` the phase.

In [ ]:
x = receive(ck.impairments.apply_polarization_mixing(tx, theta=np.deg2rad(80)))
blind = ck.equalization.cma(x, num_taps=15).signal
print(
    "as is:            ",
    np.round(ck.metrics.snr(ck.recovery.correct_carrier_phase(blind, DataAided())), 2),
    "dB",
)
ordered = ck.recovery.resolve_channel_permutation(blind)
print(
    "after permutation:",
    np.round(
        ck.metrics.snr(ck.recovery.correct_carrier_phase(ordered, DataAided())), 2
    ),
    "dB",
)

**Summary**

* A dual-polarization receiver sees a 2x2 Jones mixture; the butterfly
  equalizer inverts it with four adaptive filters.
* PMD makes the mixture frequency dependent: size `num_taps / sps` symbols
  above the DGD.
* The step size must follow the SOP's drift.
* Blind equalizers leave a permutation and a phase per output; resolve them
  against known data.

**Next:** [19 · Chromatic dispersion](19-chromatic-dispersion.ipynb).